In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

VISUALIZATION_SAMPLE_SIZE = 100000

In [ ]:
def load_simulated_bin(filepath, num_samples):
    """
    loads interleaved binary data using config constants.
    """
    filepath = Path(filepath)
    if not filepath.exists():
        return None
        
    try:
        # Determine Read Size
        # We need 2 floats per 1 complex sample
        count = num_samples * 2
        
        # Read from disk
        raw_floats = np.fromfile(filepath, dtype=SIMULATION_DTYPE, count=count)
        
        if len(raw_floats) < 2:
            print(f"Warning: {filepath.name} is too short.")
            return None
            
        i_data = raw_floats[0::2]
        q_data = raw_floats[1::2]
        
        # Ensure lengths match (in case file was odd number of bytes)
        min_len = min(len(i_data), len(q_data))
        
        # Combine
        complex_data = i_data[:min_len] + 1j * q_data[:min_len]
        
        return complex_data

    except Exception as e:
        print(f"Error reading {filepath.name}: {e}")
        return None

In [ ]:
def visualize_simulation(data, filename):
    """
    Visualizes the simulation data. 
    Uses SIMULATION_SAMPLE_RATE from config.
    """
    if data is None or len(data) == 0:
        return

    fs = SIMULATION_SAMPLE_RATE
    
    # Calculate Power Stats
    power = np.abs(data)**2
    mean_pwr = np.mean(power)
    max_pwr = np.max(power)

    fig, axs = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"File: {filename}\nMean Pwr: {mean_pwr:.4f} | Max Pwr: {max_pwr:.4f}", fontsize=14)

    # Time Domain (Zoomed)
    # 100MHz is fast, so 500 samples is a TINY slice of time (5 microseconds)
    zoom = 500 
    axs[0, 0].plot(np.real(data[:zoom]), label='I', alpha=0.8)
    axs[0, 0].plot(np.imag(data[:zoom]), label='Q', alpha=0.8)
    axs[0, 0].set_title(f'Time Domain (First {zoom} samples)')
    axs[0, 0].legend(loc='upper right')
    
    # PSD (Power Spectral Density)
    # Great for seeing the noise floor vs signal peaks
    axs[0, 1].psd(data, NFFT=1024, Fs=fs, color='purple')
    axs[0, 1].set_title('Power Spectral Density')

    # Histogram (Distribution)
    # Check for "Sticks" (Simulated) vs "Hills" (Real)
    axs[1, 0].hist(np.real(data), bins=100, alpha=0.6, label='I', color='blue')
    axs[1, 0].hist(np.imag(data), bins=100, alpha=0.6, label='Q', color='orange')
    axs[1, 0].set_title('Value Distribution')
    axs[1, 0].legend()

    # Spectrogram
    f, t_spec, Sxx = signal.spectrogram(data, fs=fs, nperseg=256, return_onesided=False)
    f = np.fft.fftshift(f)
    Sxx = np.fft.fftshift(Sxx, axes=0)
    
    axs[1, 1].pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-12), shading='gouraud', cmap='viridis')
    axs[1, 1].set_title('Spectrogram')
    axs[1, 1].set_ylabel('Frequency [Hz]')
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
search_pattern = str(SDR_CAPTURES_DIR / "*.bin")
files = sorted(glob.glob(search_pattern))

print(f"Found {len(files)} files to inspect.")
print("="*80)

for file_path in files:
    filename = os.path.basename(file_path)
    
    data = load_simulated_bin(
        file_path, 
        num_samples=VISUALIZATION_SAMPLE_SIZE
    )
    
    if data is not None:
        visualize_simulation(data, filename)
        print(f"Analyzed {filename}\n" + "-"*80)
    else:
        print(f"Skipped {filename}")